# QuantumAML · Executar o diagnóstico no Google Colab
Selecione **CPU** e **Ambiente de execução → Executar tudo**. O notebook acessa a distribuição pública v1 no Kaggle usando a máquina temporária do Colab. Precisa de aproximadamente **27 GB livres** na sessão e transfere **25,81 GB**. Os CSVs são lidos dentro do ZIP. Se a conexão cair, a célula de aquisição retoma o arquivo parcial enquanto a mesma VM existir.
    
Ao terminar, baixe **diagnostico.json**. Esse arquivo contém as medições de recursos e cobertura. As 32 janelas são distribuídas pelo arquivo de fundo de 82,88 GB; o ZIP usa compressão sequencial e alcançar janelas distantes leva tempo. A execução padrão também percorre todo o arquivo de vértices. A cobertura dos alvos depende das regiões sorteadas.

In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
required = [p for p in ("requests", "psutil", "pyarrow", "tqdm") if importlib.util.find_spec(p) is None]
if required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
import requests, shutil, time, zipfile
from tqdm.auto import tqdm
RUN_ROOT = Path("/content/quantumaml")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
URL = "https://www.kaggle.com/api/v1/datasets/download/ellipticco/elliptic2-data-set?datasetVersionNumber=1"
TOTAL = 25805701153
def obter_zip(root, url, total):
    partial = root/"elliptic2-v1.zip.part"
    final = root/"elliptic2-v1.zip"
    started = time.monotonic()
    initial = partial.stat().st_size if partial.exists() else 0
    if not final.exists():
        if shutil.disk_usage(root).free < total-initial + 1024**3:
            raise OSError("Espaço insuficiente nesta VM para o ZIP e 1 GiB de margem.")
        for attempt in range(4):
            offset = partial.stat().st_size if partial.exists() else 0
            if offset == total:
                break
            try:
                with requests.get(url, headers={"Range":f"bytes={offset}-"}, stream=True, timeout=(20,120)) as r:
                    r.raise_for_status()
                    if "/8203303/" not in r.url or (offset and r.status_code != 206):
                        raise ValueError("Versão ou retomada divergente. Confira a origem da distribuição.")
                    if r.status_code == 206 and r.headers.get("Content-Range") != f"bytes {offset}-{total-1}/{total}":
                        raise ValueError("Intervalo HTTP divergente.")
                    with partial.open("ab") as f, tqdm(total=total,initial=offset,unit="B",unit_scale=True,desc="Elliptic2 v1") as bar:
                        for block in r.iter_content(4*1024**2):
                            f.write(block)
                            bar.update(len(block))
                if partial.stat().st_size == total:
                    break
            except requests.RequestException:
                if attempt == 3:
                    raise
                time.sleep(2**attempt)
        if partial.stat().st_size != total or not zipfile.is_zipfile(partial):
            raise IOError("Download incompleto. Execute esta célula novamente na mesma sessão.")
        partial.rename(final)
    if final.stat().st_size != total:
        raise IOError("O tamanho do ZIP diverge do catálogo da v1.")
    return final, {"zip_bytes":total,"tempo_s":round(time.monotonic()-started,2),"bytes_parciais_iniciais":initial,
                   "url_publica":url}
zip_path, download_stats = obter_zip(RUN_ROOT, URL, TOTAL)

In [ ]:
"""QuantumAML 03.2. Execute com Python ou pelo notebook autocontido."""
import hashlib
import importlib
import json
import os
import platform
import random
import shutil
import sys
import tempfile
import threading
import time
from collections import Counter, defaultdict, deque
from contextlib import contextmanager
from pathlib import Path

import psutil
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as ac
import pyarrow.parquet as pq

SIZES = dict(background_edges=82877432755, background_nodes=5347932879,
             connected_components=1610070, edges=11356690, nodes=7470115)
KEYS = ["clId1", "clId2", "txId"]
MIB = 2**20


def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(MIB), b""):
            h.update(block)
    return h.hexdigest()


@contextmanager
def measure(report, name):
    proc = psutil.Process()
    rss = proc.memory_info().rss
    samples = [rss]
    stop = threading.Event()
    def monitor():
        while not stop.wait(0.025):
            samples.append(proc.memory_info().rss)
    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        stop.set()
        worker.join()
        samples.append(proc.memory_info().rss)
        report["medidas"][name] = {"segundos": round(time.perf_counter()-start, 4),
                                   "rss_inicial_MiB": round(rss/MIB, 1),
                                   "rss_max_amostrado_MiB": round(max(samples)/MIB, 1)}


def batches(path, cols=None, types=None):
    return ac.open_csv(archive.open(path) if isinstance(path,str) else path, read_options=ac.ReadOptions(block_size=MIB),
                       convert_options=ac.ConvertOptions(include_columns=cols, column_types=types))


def write_parquet(table, path):
    pq.write_table(table, path, compression="zstd")
    if not table.equals(pq.read_table(path)):
        raise ValueError(f"Falha na releitura de {path.name}")


def windows(source, target, rng, config):
    ranges = []
    with archive.open(source) as f, target.open("wb") as dst:
        header = f.readline()
        dst.write(header)
        start, end = f.tell(), archive.getinfo(source).file_size
        count = config["estratos"]
        for i in range(count):
            if i % 4 == 0:
                print(f"Estrato {i+1}/{count}", flush=True)
            lo = start + (end-start)*i//count
            hi = start + (end-start)*(i+1)//count
            # A primeira metade de cada estrato deixa espaço para linhas completas.
            offset = rng.randrange(lo, max(lo+1, (lo+hi)//2))
            f.seek(offset-1)
            if f.read(1) != b"\n":
                f.readline()
            first = f.tell()
            bytes_before = dst.tell()
            rows = 0
            while rows < config["linhas_por_janela"] and f.tell() < hi:
                line = f.readline()
                if not line or f.tell() > hi:
                    break
                if dst.tell()+len(line) > 64*MIB:
                    raise ValueError("Amostra excedeu 64 MiB")
                dst.write(line)
                rows += 1
            ranges.append(dict(inicio=first, fim_exclusivo=first+dst.tell()-bytes_before, linhas=rows))
    return ranges


def select_incident(path, ids, engine):
    if engine == "DuckDB":
        import duckdb
        with duckdb.connect(config={"threads":1,"memory_limit":"256MB"}) as conn:
            return conn.execute("SELECT clId1,clId2,txId FROM read_csv_auto(?) WHERE clId1 IN (SELECT unnest(?)) OR clId2 IN (SELECT unnest(?))",
                                [str(path),ids,ids]).fetchall()
    if engine == "Polars":
        import polars as pl
        return (pl.scan_csv(path).filter(pl.col("clId1").is_in(ids) | pl.col("clId2").is_in(ids))
                .select(KEYS).collect(engine="streaming").rows())
    selected = []
    values = pa.array(ids, type=pa.int64())
    for b in batches(path, KEYS, {k:pa.int64() for k in KEYS}):
        mask = pc.or_(pc.is_in(b[0],value_set=values),pc.is_in(b[1],value_set=values))
        selected.extend(zip(*(c.to_pylist() for c in b.filter(mask).columns)))
    return selected


def run():
    paths = {k:k+".csv" for k in SIZES}
    for k,member in paths.items():
        if archive.getinfo(member).file_size != SIZES[k]:
            raise ValueError("Tamanho divergente em "+member)
    parent = RUN_ROOT
    parent.mkdir(exist_ok=True,parents=True)
    out = Path(tempfile.mkdtemp(prefix="quantumaml-032-",dir=parent))
    interim,processed = out/"interim",out/"processed"
    interim.mkdir()
    processed.mkdir()
    cfg = {"seed":32026,"estratos":32,"linhas_por_janela":1000,"max_vertices":12,"limite_sementes":512}
    free = shutil.disk_usage(out)
    if free.free < 96*MIB:
        raise OSError("São necessários 96 MiB livres para os arquivos de trabalho.")
    ram = psutil.virtual_memory()
    report = {"fonte":"ellipticco/elliptic2-data-set","versao_esperada":1,"prefixos_locais":False,"aquisicao":download_stats,
              "config":cfg,"python":sys.version.split()[0],"sistema":platform.system(),"cpus_logicas":os.cpu_count(),
              "afinidade_cpu":len(os.sched_getaffinity(0)) if hasattr(os,"sched_getaffinity") else None,
              "ram_total":ram.total,"ram_disponivel":ram.available,"disco_total":free.total,"disco_livre":free.free,
              "libs":{},"entradas":{},"medidas":{}}
    for key in ("cpu.max","memory.max","memory.current"):
        control = Path("/sys/fs/cgroup")/key
        if control.is_file():
            report[key] = control.read_text().strip()
    for lib in ("pyarrow","polars","duckdb","psutil"):
        try:
            report["libs"][lib] = importlib.import_module(lib).__version__
        except ImportError:
            report["libs"][lib] = None
    rng = random.Random(cfg["seed"])
    for k in paths:
        report["entradas"][k] = {"arquivo":paths[k],"bytes":SIZES[k]}
    sample = interim/"janelas.csv"
    with measure(report,"amostragem_arestas"):
        report["intervalos"] = windows(paths["background_edges"],sample,rng,cfg)
        edges = []
        for b in batches(sample,KEYS):
            edges.extend(zip(*(a.to_pylist() for a in b.columns)))
    adj = defaultdict(set)
    vertices = set()
    for u,v,t in edges:
        adj[u].add(v)
        vertices.update((u,v))
    seeds = rng.sample(sorted(adj),min(cfg["limite_sementes"],len(adj)))
    query = sorted(set(map(int,os.environ["QUANTUMAML_IDS"].split(",")))) if os.getenv("QUANTUMAML_IDS") else sorted(seeds[:16])
    if len(query)>512:
        raise ValueError("A consulta admite até 512 IDs.")
    report["consulta_ids"] = query
    ref = sorted(t for t in edges if t[0] in query or t[1] in query)
    for label, lib in [("Arrow","pyarrow"),("DuckDB","duckdb"),("Polars","polars")]:
        if report["libs"][lib]:
            with measure(report,"incidentes_"+label):
                answer = select_incident(sample,query,label)
            if sorted(answer) != ref:
                raise ValueError("Divergência entre métodos: "+label)
    with measure(report,"geracao_e_parquet"):
        groups = set()
        for seed in seeds:
            visited, queue = {seed},deque([seed])
            while queue and len(visited)<cfg["max_vertices"]:
                for v in sorted(adj.get(queue.popleft(),())):
                    if len(visited)==cfg["max_vertices"]:
                        break
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
            if len(visited)>1:
                groups.add(tuple(sorted(visited)))
        groups = sorted(groups)
        groupfile = processed/"candidatos.parquet"
        write_parquet(pa.table({"membros":pa.array(groups,type=pa.list_(pa.int64()))}),groupfile)
        report["hash_candidatos_antes_rotulos"] = sha(groupfile)
        for filename, rows in [("arestas_regioes",edges),("incidentes_consulta",ref)]:
            write_parquet(pa.table({k:pa.array([t[i] for t in rows], type=pa.int64()) for i,k in enumerate(KEYS)}),
                          interim/(filename+".parquet"))
    with measure(report,"consulta_vertices"):
        matches = []
        query_array = pa.array(query,type=pa.int64())
        # Preservar as representações originais mesmo se houver campos mistos mais adiante.
        node_types = {f"feat#{i}":pa.string() for i in range(1,44)}
        node_reader = batches(paths["background_nodes"],types={**node_types,"clId":pa.int64()})
        for b in node_reader:
            keep = b.filter(pc.is_in(b.column("clId"),value_set=query_array))
            if keep.num_rows:
                matches.append(keep)
        write_parquet(pa.Table.from_batches(matches,schema=node_reader.schema),interim/"vertices_consulta.parquet")
    if os.getenv("QUANTUMAML_INCIDENCIA_COMPLETA") == "1":
        with measure(report,"incidencia_arquivo_inteiro"):
            stored, total, scanned = [], 0, 0
            reader = batches(paths["background_edges"], KEYS, {k:pa.int64() for k in KEYS})
            for b in reader:
                scanned += b.num_rows
                hit = b.filter(pc.or_(pc.is_in(b[0],value_set=query_array),pc.is_in(b[1],value_set=query_array)))
                if hit.num_rows and total < 10000:
                    stored.append(hit.slice(0,10000-total))
                total += hit.num_rows
            write_parquet(pa.Table.from_batches(stored,schema=reader.schema),interim/"incidentes_arquivo.parquet")
            report["incidencia_arquivo_inteiro"] = {"examinadas":scanned,"ocorrencias":total,"limite_exportacao":10000}
    # Consultar as anotações somente depois da gravação e do hash dos candidatos.
    with measure(report,"avaliacao"):
        labels, owner = {}, {}
        sizes = Counter()
        present = Counter()
        for b in batches(paths["connected_components"],["ccId","ccLabel"]):
            labels.update(zip(*(c.to_pylist() for c in b.columns)))
        for b in batches(paths["nodes"],["clId","ccId"]):
            for u,g in zip(*(c.to_pylist() for c in b.columns)):
                sizes[g]+=1
                if u in vertices:
                    owner[u]=g
                    present[g]+=1
        covered = set()
        for group in groups:
            for g,count in Counter(owner[u] for u in group if u in owner).items():
                if count==sizes[g]:
                    covered.add(g)
        report["cobertura"] = {}
        for label in sorted(set(labels.values())):
            report["cobertura"][label] = dict(total=sum(v==label for v in labels.values()),
                parcial_exclusiva=sum(labels[g]==label and n<sizes[g] for g,n in present.items()),
                integral_disponivel=sum(labels[g]==label and n==sizes[g] for g,n in present.items()),
                integral_em_candidato=sum(labels[g]==label for g in covered))
    for k,p in paths.items():
        report["entradas"][k]["esquema_inferido_primeiro_bloco"] = str(batches(p).schema)
    report.update(arestas_lidas=len(edges),vertices_nas_janelas=len(vertices),sementes=len(seeds),
                  candidatos=len(groups),incidentes=len(ref),vertices_consulta_encontrados=sum(b.num_rows for b in matches),
                  bytes_janelas=sample.stat().st_size)
    report["parquet"] = {p.name:{"bytes":p.stat().st_size,"linhas":pq.read_metadata(p).num_rows,"sha256":sha(p)}
                         for p in out.rglob("*.parquet")}
    report["disco_livre_final"] = shutil.disk_usage(out).free
    (out/"diagnostico.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf8")
    print(out)
    print(json.dumps({k:report[k] for k in ("libs","arestas_lidas","candidatos","cobertura","medidas")},indent=2))
    return out/"diagnostico.json"




In [ ]:
with zipfile.ZipFile(zip_path) as archive:
    resultado = run()
print("Relatório salvo em", resultado)

In [ ]:
from google.colab import files
files.download(str(resultado))